# Hyperformer για WLASL-100


**Βήματα:**
1. Ανέβασε τα .npy αρχεία
2. Ανέβασε το Hyperformer-main.zip
3. Τρέξε τα cells με σειρά

In [ ]:
#Ανέβασε τα αρχεία
from google.colab import files
import os

print('Ανέβασε το Hyperformer-main.zip')
uploaded = files.upload()

Ανέβασε το Hyperformer-main.zip


Saving Hyperformer-main.zip to Hyperformer-main.zip


In [ ]:
# Ανέβασε τα .npy data
print('Ανέβασε τα 6 .npy αρχεία (X_train, X_val, X_test, y_train, y_val, y_test)')
uploaded_data = files.upload()

Ανέβασε τα 6 .npy αρχεία (X_train, X_val, X_test, y_train, y_val, y_test)


Saving X_val.npy to X_val.npy
Saving y_test.npy to y_test.npy
Saving y_train.npy to y_train.npy
Saving y_val.npy to y_val.npy
Saving X_train.npy to X_train.npy
Saving X_test.npy to X_test.npy


In [ ]:
# Cell 4: Setup φακέλων
import zipfile, shutil

# Extract Hyperformer
with zipfile.ZipFile('Hyperformer-main.zip', 'r') as z:
    z.extractall('.')

os.chdir('Hyperformer-main')

# Δημιουργία φακέλων για data
os.makedirs('data/WLASL_npy_dataset_100_split', exist_ok=True)

# Μεταφορά .npy αρχείων
for fname in ['X_train.npy','X_val.npy','X_test.npy','y_train.npy','y_val.npy','y_test.npy']:
    src = f'/content/{fname}'
    if os.path.exists(src):
        shutil.copy(src, f'data/WLASL_npy_dataset_100_split/{fname}')
        print(f'✅  {fname}')
    else:
        print(f' {fname} δεν βρέθηκε')

print('\nΤρέχον directory:', os.getcwd())

✅ X_train.npy
✅ X_val.npy
✅ X_test.npy
✅ y_train.npy
✅ y_val.npy
✅ y_test.npy

Τρέχον directory: /content/Hyperformer-main


In [ ]:
# Cell 5: Δημιουργία feeder_wlasl.py
feeder_code = '''
import numpy as np
from torch.utils.data import Dataset

class Feeder(Dataset):
    def __init__(self, data_path, label_path, split="train",
                 bone=False, vel=False, debug=False, **kwargs):
        self.bone = bone
        self.vel  = vel
        X = np.load(data_path).astype(np.float32)
        y = np.load(label_path).astype(np.int64)
        if debug:
            X, y = X[:100], y[:100]
        N, T, F = X.shape
        V = F // 3
        X = X.reshape(N, T, V, 3).transpose(0, 3, 1, 2)[:, :, :, :, np.newaxis]
        self.data  = X
        self.label = y
        self.sample_name = [f"{split}_{i}" for i in range(N)]
        print(f"[Feeder-WLASL] {split}: {N} samples | shape={X.shape}")

    def __len__(self): return len(self.label)

    def __getitem__(self, idx):
        data  = np.array(self.data[idx])
        label = int(self.label[idx])
        if self.vel:
            data[:, :-1] = data[:, 1:] - data[:, :-1]
            data[:, -1]  = 0
        return data, label, idx
'''
with open('feeders/feeder_wlasl.py', 'w') as f:
    f.write(feeder_code)
print('✅ feeder_wlasl.py')

✅ feeder_wlasl.py


In [ ]:
# Cell 6: Δημιουργία mediapipe_holistic.py
graph_code = '''
import sys, numpy as np
sys.path.extend(["../"])
from graph import tools

num_node = 75
self_link = [(i, i) for i in range(num_node)]
pose_edges = [
    (0,1),(1,2),(2,3),(3,7),(0,4),(4,5),(5,6),(6,8),(9,10),
    (11,12),(11,13),(13,15),(12,14),(14,16),(11,23),(12,24),
    (23,24),(23,25),(25,27),(27,29),(24,26),(26,28),(28,30),
    (15,17),(15,19),(16,18),(16,20),
]
hand_base = [
    (0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),(0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20),(5,9),(9,13),(13,17),
]
lh = [(a+33, b+33) for a,b in hand_base]
rh = [(a+54, b+54) for a,b in hand_base]
wrist = [(15,33),(16,54)]
inward  = pose_edges + lh + rh + wrist
outward = [(j,i) for i,j in inward]
neighbor = inward + outward

class Graph:
    def __init__(self, labeling_mode="spatial", scale=1):
        self.num_node = num_node
        self.self_link = self_link
        self.inward = inward
        self.outward = outward
        self.neighbor = neighbor
        self.A = self.get_adjacency_matrix(labeling_mode)
        self.A_binary   = tools.edge2mat(neighbor, num_node)
        self.A_norm     = tools.normalize_adjacency_matrix(self.A_binary + 2*np.eye(num_node))
        self.A_binary_K = tools.get_k_scale_graph(scale, self.A_binary)
        self.A1 = self.A; self.A2 = self.A
        self.A_A1 = self.A; self.A1_A2 = self.A

    def get_adjacency_matrix(self, labeling_mode=None):
        return tools.get_spatial_graph(num_node, self_link, inward, outward)
'''
with open('graph/mediapipe_holistic.py', 'w') as f:
    f.write(graph_code)

# __init__.py για graph
with open('graph/__init__.py', 'a') as f:
    f.write('\nfrom graph import mediapipe_holistic\n')
print('✅ mediapipe_holistic.py')

✅ mediapipe_holistic.py


In [ ]:
# Cell 7: Config YAML
import yaml

config = {
    'work_dir': './work_dir/wlasl100',
    'feeder': 'feeders.feeder_wlasl.Feeder',
    'train_feeder_args': {
        'data_path': 'data/WLASL_npy_dataset_100_split/X_train.npy',
        'label_path': 'data/WLASL_npy_dataset_100_split/y_train.npy',
        'split': 'train', 'bone': False, 'vel': False, 'debug': False
    },
    'test_feeder_args': {
        'data_path': 'data/WLASL_npy_dataset_100_split/X_test.npy',
        'label_path': 'data/WLASL_npy_dataset_100_split/y_test.npy',
        'split': 'test', 'bone': False, 'vel': False
    },
    'model': 'model.Hyperformer.Model',
    'model_args': {
        'num_class': 100, 'num_point': 75, 'num_person': 1, 'num_of_heads': 6,
        'graph': 'graph.mediapipe_holistic.Graph',
        'graph_args': {'labeling_mode': 'spatial'},
        'joint_label': [0,0,0,0,0,0,0,0,0,0,0,1,2,1,2,1,2,1,2,1,2,1,2,3,3,4,5,4,5,4,5,4,5,
                        6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,
                        7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7]
    },
    'num_epoch': 50, 'batch_size': 16, 'test_batch_size': 32,
    'base_lr': 0.0003, 'lr_decay_rate': 0.1, 'step': [35, 45],
    'optimizer': 'AdamW', 'weight_decay': 0.0004, 'nesterov': False,
    'device': [0], 'num_worker': 0, 'seed': 42,
    'warm_up_epoch': 0, 'save_score': True
}

os.makedirs('config/wlasl100', exist_ok=True)
with open('config/wlasl100/joint.yaml', 'w') as f:
    yaml.dump(config, f)
print('✅ config/wlasl100/joint.yaml')

✅ config/wlasl100/joint.yaml


In [ ]:
# Cell 8: Patch main.py για Linux compatibility
with open('main.py', 'r') as f:
    code = f.read()

# Fix yaml.load
code = code.replace('yaml.load(f)', 'yaml.load(f, Loader=yaml.SafeLoader)')

# Fix prefetch_factor
code = code.replace('prefetch_factor=16,', 'prefetch_factor=4,')

# Fix model.module.num_class
code = code.replace('self.model.module.num_class', 'self.model.num_class')

# Fix AdamW in adjust_learning_rate
code = code.replace(
    "if self.arg.optimizer == 'SGD' or self.arg.optimizer == 'Adam' or self.arg.optimizer == 'NAdam':",
    "if self.arg.optimizer == 'SGD' or self.arg.optimizer == 'Adam' or self.arg.optimizer == 'NAdam' or self.arg.optimizer == 'AdamW':"
)

with open('main.py', 'w') as f:
    f.write(code)
print('✅ main.py patched')

✅ main.py patched


In [ ]:

import subprocess
subprocess.run(['pip', 'install', '-e', 'torchlight'], cwd='/content/Hyperformer-main')

CompletedProcess(args=['pip', 'install', '-e', 'torchlight'], returncode=0)

In [ ]:
!pip install torchpack

In [ ]:
import sys
sys.path.insert(0, '/content/Hyperformer-main/torchlight')

torchlight_init = '/content/Hyperformer-main/torchlight/torchlight/__init__.py'
with open(torchlight_init, 'w') as f:
    f.write('''
import argparse

class DictAction(argparse.Action):
    def __call__(self, parser, namespace, values, option_string=None):
        setattr(namespace, self.dest, dict(values))

class IO:
    pass
''')
print('✅ torchlight patched')

✅ torchlight patched


In [ ]:
# feeder_wlasl.py
feeder_code = '''
import numpy as np
from torch.utils.data import Dataset

class Feeder(Dataset):
    def __init__(self, data_path, label_path, split="train",
                 bone=False, vel=False, debug=False, **kwargs):
        self.bone = bone
        self.vel  = vel
        X = np.load(data_path).astype(np.float32)
        y = np.load(label_path).astype(np.int64)
        if debug:
            X, y = X[:100], y[:100]
        N, T, F = X.shape
        V = F // 3
        X = X.reshape(N, T, V, 3).transpose(0, 3, 1, 2)[:, :, :, :, np.newaxis]
        self.data  = X
        self.label = y
        self.sample_name = [f"{split}_{i}" for i in range(N)]
        print(f"[Feeder-WLASL] {split}: {N} samples | shape={X.shape}")

    def __len__(self): return len(self.label)

    def __getitem__(self, idx):
        data  = np.array(self.data[idx])
        label = int(self.label[idx])
        if self.vel:
            data[:, :-1] = data[:, 1:] - data[:, :-1]
            data[:, -1]  = 0
        return data, label, idx

    def top_k(self, score, top_k):
        rank = score.argsort()
        hit_top_k = [l in rank[i, -top_k:] for i, l in enumerate(self.label)]
        return sum(hit_top_k) / len(hit_top_k)
'''

with open('/content/Hyperformer-main/feeders/feeder_wlasl.py', 'w') as f:
    f.write(feeder_code)
print('✅ feeder_wlasl.py updated με top_k')

✅ feeder_wlasl.py updated με top_k


In [ ]:
import shutil, os
if os.path.exists('./work_dir/wlasl100/runs'):
    shutil.rmtree('./work_dir/wlasl100/runs')
    print('✅ Διαγράφηκε')

In [ ]:
#prefetch_factor στο main.py
with open('main.py', 'r') as f:
    code = f.read()

code = code.replace('prefetch_factor=16,', 'prefetch_factor=None,')
code = code.replace('prefetch_factor=4,', 'prefetch_factor=None,')

with open('main.py', 'w') as f:
    f.write(code)
print('✅ prefetch_factor fixed')

✅ prefetch_factor fixed


In [ ]:
#Train
import subprocess, sys

process = subprocess.Popen(
    ['python', 'main.py', '--config', 'config/wlasl100/joint.yaml'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end='', flush=True)

process.wait()

Streaming output truncated to the last 5000 lines.
100%|███| 88/88 [01:02<00:00,  1.42it/s]
[ Mon May  4 22:29:47 2026 ] 	Mean training loss: 3.0417. loss2: 0.0000. Mean training acc: 26.42%.
[ Mon May  4 22:29:47 2026 ] 	Time consumption: [Data]00%, [Network]99%
[ Mon May  4 22:29:47 2026 ] Eval epoch: 7

100%|█████| 8/8 [00:05<00:00,  1.46it/s]
Accuracy:  0.20080321285140562  model:  ./work_dir/wlasl100/runs
[ Mon May  4 22:29:52 2026 ] 	Mean test loss of 8 batches: 3.411526709794998.
[ Mon May  4 22:29:52 2026 ] 	Top1: 20.08%
[ Mon May  4 22:29:52 2026 ] 	Top5: 46.18%
[ Mon May  4 22:29:52 2026 ] Training epoch: 8

  0%|            | 0/88 [00:00<?, ?it/s]/content/Hyperformer-main/main.py:469: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/content/Hyperformer-main/main.py:497: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use

0

In [ ]:
#Aποτελέσματα
!cat work_dir/wlasl100/log.txt | grep -E 'Top|Accuracy|epoch' | tail -20

[ Mon May  4 23:12:38 2026 ] Training epoch: 46
[ Mon May  4 23:13:40 2026 ] Eval epoch: 46
[ Mon May  4 23:13:45 2026 ] 	Top1: 46.59%
[ Mon May  4 23:13:45 2026 ] 	Top5: 73.49%
[ Mon May  4 23:13:45 2026 ] Training epoch: 47
[ Mon May  4 23:14:47 2026 ] Eval epoch: 47
[ Mon May  4 23:14:53 2026 ] 	Top1: 46.18%
[ Mon May  4 23:14:53 2026 ] 	Top5: 72.69%
[ Mon May  4 23:14:53 2026 ] Training epoch: 48
[ Mon May  4 23:15:55 2026 ] Eval epoch: 48
[ Mon May  4 23:16:00 2026 ] 	Top1: 46.18%
[ Mon May  4 23:16:00 2026 ] 	Top5: 71.89%
[ Mon May  4 23:16:00 2026 ] Training epoch: 49
[ Mon May  4 23:17:02 2026 ] Eval epoch: 49
[ Mon May  4 23:17:08 2026 ] 	Top1: 47.39%
[ Mon May  4 23:17:08 2026 ] 	Top5: 71.89%
[ Mon May  4 23:17:08 2026 ] Training epoch: 50
[ Mon May  4 23:18:09 2026 ] Eval epoch: 50
[ Mon May  4 23:18:15 2026 ] 	Top1: 46.99%
[ Mon May  4 23:18:15 2026 ] 	Top5: 73.09%


In [ ]:
with open('./work_dir/wlasl100/epoch49_test_score.pkl', 'rb') as f:
    scores = pickle.load(f)

score_array = np.array([scores[f'test_{i}'] for i in range(len(labels))])

top1  = sum([l == np.argsort(s)[-1]   for s, l in zip(score_array, labels)]) / len(labels)
top5  = sum([l in np.argsort(s)[-5:]  for s, l in zip(score_array, labels)]) / len(labels)
top10 = sum([l in np.argsort(s)[-10:] for s, l in zip(score_array, labels)]) / len(labels)

print(f"Top-1:  {top1*100:.2f}%")
print(f"Top-5:  {top5*100:.2f}%")
print(f"Top-10: {top10*100:.2f}%")

Top-1:  47.39%
Top-5:  71.89%
Top-10: 82.73%


## **B. Hyperformer με Proxy Anchor στο wlasl100**

In [ ]:
# Patch Hyperformer.py - επιστροφή embedding πριν το fc
with open('model/Hyperformer.py', 'r') as f:
    code = f.read()

code = code.replace(
    '''        x = self.drop_out(x)
        x = self.fc(x)

        return x, y''',
    '''        x = self.drop_out(x)
        emb = x  # embedding πριν το fc
        x = self.fc(x)

        return x, emb'''
)

with open('model/Hyperformer.py', 'w') as f:
    f.write(code)
print(' Hyperformer.py patched - επιστρέφει embedding')

✅ Hyperformer.py patched - επιστρέφει embedding


In [ ]:
with open('main.py', 'r') as f:
    code = f.read()

proxy_anchor_code = '''
class ProxyAnchorLoss(torch.nn.Module):
    def __init__(self, num_classes, emb_dim, margin=0.1, alpha=32):
        super().__init__()
        self.proxies = torch.nn.Parameter(torch.randn(num_classes, emb_dim))
        torch.nn.init.kaiming_normal_(self.proxies, mode="fan_out")
        self.num_classes = num_classes
        self.margin = margin
        self.alpha = alpha

    def forward(self, X, labels):
        import torch.nn.functional as F
        P = F.normalize(self.proxies, dim=1)
        X = F.normalize(X, dim=1)
        cos = X @ P.T
        one_hot = torch.zeros(len(labels), self.num_classes, device=X.device)
        one_hot.scatter_(1, labels.unsqueeze(1), 1.0)
        pos_exp = torch.exp(-self.alpha * (cos - self.margin))
        neg_exp = torch.exp(self.alpha * (cos + self.margin))
        with_pos = (one_hot.sum(0) > 0)
        P_sum = (one_hot * pos_exp).sum(0)
        N_sum = ((1 - one_hot) * neg_exp).sum(0)
        pos_term = torch.log(1 + P_sum)[with_pos].sum() / with_pos.sum().clamp(min=1)
        neg_term = torch.log(1 + N_sum).sum() / self.num_classes
        return pos_term + neg_term
'''

code = code.replace(
    'class Processor(object):',
    proxy_anchor_code + '\nclass Processor(object):'
)

# Αντικατάσταση loss - χρησιμοποιεί emb αντί output
code = code.replace(
    'self.loss = nn.CrossEntropyLoss()',
    'self.loss = nn.CrossEntropyLoss()\n        self.proxy_loss = ProxyAnchorLoss(num_classes=100, emb_dim=144)'
)

# Αντικατάσταση forward
code = code.replace(
    'output, z = self.model(data, F.one_hot(label, num_classes=self.model.num_class))',
    'output, emb = self.model(data, F.one_hot(label, num_classes=self.model.num_class))\n                z = emb'
)

# Loss = CrossEntropy + Proxy Anchor
code = code.replace(
    'loss = self.loss(output, label)',
    'loss = self.loss(output, label) + self.proxy_loss(emb, label)'
)

with open('main.py', 'w') as f:
    f.write(code)
print('✅ main.py patched - Proxy Anchor Loss added')

✅ main.py patched - Proxy Anchor Loss added


In [ ]:
import shutil, os
if os.path.exists('./work_dir/wlasl100/runs'):
    shutil.rmtree('./work_dir/wlasl100/runs')
print('✅ Έτοιμο')

✅ Έτοιμο


In [ ]:
with open('main.py', 'r') as f:
    lines = f.readlines()

# Διόρθωση γραμμών 577-579
lines[576] = '                    output, emb = self.model(data, F.one_hot(label, num_classes=self.model.num_class))\n'
lines[577] = '                    z = emb\n'
lines[578] = '\n'  # αδειάζω την παλιά γραμμή

with open('main.py', 'w') as f:
    f.writelines(lines)
print('✅ Fixed')

✅ Fixed


In [ ]:
with open('main.py', 'r') as f:
    code = f.read()

proxy_class = '''
import torch.nn.functional as F_pa

class ProxyAnchorLoss(torch.nn.Module):
    def __init__(self, num_classes, emb_dim, margin=0.1, alpha=32):
        super().__init__()
        self.proxies = torch.nn.Parameter(torch.randn(num_classes, emb_dim))
        torch.nn.init.kaiming_normal_(self.proxies, mode="fan_out")
        self.num_classes = num_classes
        self.margin = margin
        self.alpha = alpha

    def forward(self, X, labels):
        P = F_pa.normalize(self.proxies, dim=1)
        X = F_pa.normalize(X, dim=1)
        cos = X @ P.T
        one_hot = torch.zeros(len(labels), self.num_classes, device=X.device)
        one_hot.scatter_(1, labels.unsqueeze(1), 1.0)
        pos_exp = torch.exp(-self.alpha * (cos - self.margin))
        neg_exp = torch.exp(self.alpha * (cos + self.margin))
        with_pos = (one_hot.sum(0) > 0)
        P_sum = (one_hot * pos_exp).sum(0)
        N_sum = ((1 - one_hot) * neg_exp).sum(0)
        pos_term = torch.log(1 + P_sum)[with_pos].sum() / with_pos.sum().clamp(min=1)
        neg_term = torch.log(1 + N_sum).sum() / self.num_classes
        return pos_term + neg_term

'''

# Βάλε το class αμέσως μετά το "import torch"
code = code.replace('import torch\n', 'import torch\n' + proxy_class, 1)

with open('main.py', 'w') as f:
    f.write(code)

# Επιβεβαίωση
print('ProxyAnchorLoss in main.py:', 'ProxyAnchorLoss' in open('main.py').read())

ProxyAnchorLoss in main.py: True


In [ ]:
import subprocess

process = subprocess.Popen(
    ['python', 'main.py', '--config', 'config/wlasl100/joint.yaml'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end='', flush=True)

process.wait()

Streaming output truncated to the last 5000 lines.
100%|███| 88/88 [01:02<00:00,  1.41it/s]
[ Mon May  4 23:44:04 2026 ] 	Mean training loss: 11.9754. loss2: 0.0000. Mean training acc: 17.76%.
[ Mon May  4 23:44:04 2026 ] 	Time consumption: [Data]00%, [Network]99%
[ Mon May  4 23:44:04 2026 ] Eval epoch: 7

100%|█████| 8/8 [00:10<00:00,  1.37s/it]
Accuracy:  0.17269076305220885  model:  ./work_dir/wlasl100/runs
[ Mon May  4 23:44:15 2026 ] 	Mean test loss of 8 batches: 12.93336033821106.
[ Mon May  4 23:44:15 2026 ] 	Top1: 17.27%
[ Mon May  4 23:44:15 2026 ] 	Top5: 39.36%
[ Mon May  4 23:44:15 2026 ] Training epoch: 8

  0%|            | 0/88 [00:00<?, ?it/s]/content/Hyperformer-main/main.py:497: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/content/Hyperformer-main/main.py:525: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please us

0

In [ ]:
import numpy as np, pickle

# Βρες το best epoch
with open('./work_dir/wlasl100/epoch36_test_score.pkl', 'rb') as f:
    scores = pickle.load(f)

labels = np.load('data/WLASL_npy_dataset_100_split/y_test.npy')
score_array = np.array([scores[f'test_{i}'] for i in range(len(labels))])

top1  = sum([l == np.argsort(s)[-1]   for s, l in zip(score_array, labels)]) / len(labels)
top5  = sum([l in np.argsort(s)[-5:]  for s, l in zip(score_array, labels)]) / len(labels)
top10 = sum([l in np.argsort(s)[-10:] for s, l in zip(score_array, labels)]) / len(labels)

print(f"Top-1:  {top1*100:.2f}%")
print(f"Top-5:  {top5*100:.2f}%")
print(f"Top-10: {top10*100:.2f}%")

Top-1:  49.00%
Top-5:  75.50%
Top-10: 83.13%
